<a href="https://colab.research.google.com/github/kimhozzi/Kleague_Crawling/blob/main/crawl_indi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
## CAUTION!  주피터 노트북 기준으로 함
## git에 올리지만 readme에 주피터 노트북에서 돌리라고 해라.
### 이걸로 페이지 만들려면 DB화는 필수일듯 ai 돌리는게 아니라서 ㅇㅇ

!pip install beautifulsoup4 requests selenium lxml openpyxl

In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from bs4 import BeautifulSoup
import pandas as pd

# 웹드라이버를 설정합니다. (여기서는 Chrome을 사용합니다.)
# 웹드라이버의 경로를 자신의 시스템에 맞게 설정해주세요.
options = Options()
options.page_load_strategy = 'eager'
driver = webdriver.Chrome(options=options)

# 웹사이트에 접속합니다.
driver.get('https://data.kleague.com')

# Find frameset element and get its children
frames = driver.find_elements("tag name", 'frame')

for frame in frames:
    if 'https://portal.kleague.com' in frame.get_attribute('src'):
        # redirect the browser to the frame url
        driver.get(frame.get_attribute('src'))
    else:
        # do nothing
        pass

In [ ]:
import re

def extract_onclick_data(onclick):
    if "moveMainFrameMcPlayer" in onclick:
        match = re.search(r"moveMainFrameMcPlayer\('(.+?)','(.+?)','(.+?)'\)", onclick)
        if match:
            return dict(zip(['menuCd', 'playerId', 'teamId'], match.groups()))
    elif "moveMainFrame" in onclick:
        match = re.search(r"moveMainFrame\('(.+?)'\)", onclick)
        if match:
            return {'menuCd': match.group(1)}
    return {}

In [ ]:
# JavaScript를 실행하여 페이지를 이동합니다.
# driver.execute_script("moveMainFrame('0011')")
### !! click 아닌 excute_script 사용 가능성 물어보기
driver.execute_script("moveMainFrame('0415')")

In [ ]:
# 페이지의 HTML을 가져옵니다.
html = driver.page_source

# BeautifulSoup 객체를 생성합니다.
soup = BeautifulSoup(html, 'html.parser')


# 특정 form 안에 있는 모든 테이블을 찾아 데이터를 스크랩핑합니다.
form = soup.find('form', {'id': 'frm'})
table_data = []
if form is not None:
    tables = form.find_all('table')
    for table in tables:
        # Get column names from thead
        column_names = [col.text for col in table.thead.find_all('th')]
        # Get all rows in the table
        rows = table.find_all('tr')
        row_data = []
        for row in rows:
            # Get all columns in the row
            cols = row.find_all('td')
            # Get onclick attribute value
            onclick = row.get('onclick')
            # Append the onclick value to the row data if it exists
            cols_data = dict(zip(column_names, [col.text for col in cols]))
            if onclick is not None:
                cols_data.update(extract_onclick_data(onclick))
            row_data.append(cols_data)
        table_data.append(pd.DataFrame(row_data))

# 데이터를 출력합니다.
for data in table_data:
    print(data)

# # Export data to excel
# with pd.ExcelWriter('kl_data.xlsx') as writer:
#     for i, data in enumerate(table_data):

       소속 클럽   등번   성명      키   몸무게        생년월일 menuCd  playerId teamId
0        NaN  NaN  NaN    NaN   NaN         NaN    NaN       NaN    NaN
1       강원FC   41  김유성  187Cm  72Kg  2005/11/16   0416  20240099    K21
2       강원FC   21  박청효  190Cm  78Kg  1990/02/13   0416  20130134    K21
3       강원FC    1  이광연  184Cm  85Kg  1999/09/11   0416  20190058    K21
4       강원FC   71  조민규  193Cm  87Kg  2003/04/30   0416  20230068    K21
..       ...  ...  ...    ...   ...         ...    ...       ...    ...
103  포항 스틸러스   21  황인재  187Cm  73Kg  1994/04/22   0416  20160079    K03
104     화성FC   21  강성국  185Cm  76Kg  2005/03/11   0416  20250173    K39
105     화성FC   18  김기훈  186Cm  84Kg  2002/11/14   0416  20250172    K39
106     화성FC    1  김승건  189Cm  82Kg  1999/02/08   0416  20250166    K39
107     화성FC   13  이기현  192Cm  84Kg  1993/12/16   0416  20150056    K39

[108 rows x 9 columns]
    소속 클럽   등번   성명      키   몸무게        생년월일 menuCd  playerId teamId
0     NaN  NaN  NaN    NaN   NaN         Na

In [ ]:
#선수 이름을 검색하여 해당 선수의 playerId와 teamId를 찾고, 이 정보를 사용하여 페이지를 이동시키는 코드를 추가
# Search for a specific player
player_name = "최경록"  # replace with the name of the player you are looking for  이거 for문으로 자동으로 내리기
for data in table_data:
    player_data = data[data['성명'] == player_name]
    if not player_data.empty:
        player_id = player_data['playerId'].values[0]
        team_id = player_data['teamId'].values[0]
        # Move to the page with the player's details
        driver.execute_script(f"moveMainFrameMcPlayer('0416', '{player_id}', '{team_id}')")
        break


In [ ]:
import pandas as pd

# Execute JavaScript to get the value of resultDataSet
result_data_set = driver.execute_script("return resultDataSet;")

# Check if resultDataSet is a list
if isinstance(result_data_set, list):
    print(f"resultDataSet contains {len(result_data_set)} objects.")
    for i, obj in enumerate(result_data_set):
        # Convert the object to a DataFrame
        df = pd.DataFrame(obj)
        # Print the DataFrame
        print(f"Object {i+1}:")
        print(df)
else:
    print(f"resultDataSet is not a list. It is a {type(result_data_set)}.")

# Export data to excel with name of player_name as sheet name
with pd.ExcelWriter('kl_player_data.xlsx') as writer:
    for i, obj in enumerate(result_data_set):
        # Convert the object to a DataFrame
        df = pd.DataFrame(obj)
        df.to_excel(writer, sheet_name=f"{player_name}_{i+1}", index=False)

# 추후 csv 파일로 저장하게 하기

resultDataSet contains 7 objects.
Object 1:
   Team_S_Name Team_id
0           강원     K21
1           광주     K22
2           김천     K35
3           대구     K17
4           대전     K10
5           서울     K09
6         수원FC     K29
7           안양     K27
8           울산     K01
9           전북     K05
10          제주     K04
11          포항     K03
Object 2:
     game_date  game_id game_time  meet_seq meet_year  round_id team_id  \
0   2025/02/15        3     16:30         1      2025         1     K22   
1   2025/02/23       11     16:30         1      2025         2     K22   
2   2025/03/01       15     16:30         1      2025         3     K22   
3   2025/03/22       23     16:30         1      2025         4     K22   
4   2025/03/16       29     16:30         1      2025         5     K22   
5   2025/03/29       33     16:30         1      2025         6     K22   
6   2025/04/06       42     16:30         1      2025         7     K22   
7   2025/04/13       46     14:00         1    